# Parse outputs -> tidy tables

Reads the raw moral-generation output (canonical **`moral_generation_plus.jsonl`**,
or falls back to the `output_json` column of `moral_generation_plus.csv`) and
derives flat, analysis-ready CSVs into `tidy/`, each keyed by a stable
`generation_id` (`filename__input_condition__model`).

**Active by default:**
- **`morals_long.csv`** - one row per story moral.
- **`categories_wide.csv`** - one row per generation, one column per top-level
  schema category. `protagonist` is reduced to its `name`; every other category
  (which carries multiple answers/fields) is written as a JSON string.

**Optional (commented out in section 3 - un-comment to enable):**
`dimensions_wide`, `values_long`, `themes_long`, `actions_long`, `locations_long`
- per-dimension tables, one row per item, joinable via `generation_id`.

## 1. Imports + parameters

In [1]:
import json
from pathlib import Path
import pandas as pd

RAW_JSONL = Path("moral_generation_plus.jsonl")   # canonical raw store
RAW_CSV   = Path("moral_generation_plus.csv")     # fallback (uses output_json column)
OUTDIR    = Path("tidy")                          # where the tidy CSVs are written
OUTDIR.mkdir(exist_ok=True)

## 2. Load the raw records

Each record is a dict with `filename`, `input_condition`, `model`, and `result`
(the parsed structured object, or None if the model's reply did not parse).

In [3]:
def load_records():
    # Prefer the canonical JSONL.
    if RAW_JSONL.exists():
        recs = []
        with open(RAW_JSONL, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    recs.append(json.loads(line))
        print(f"Loaded {len(recs)} record(s) from {RAW_JSONL}")
        return recs
    # Fallback: a CSV that still carries the full JSON in an `output_json` column.
    if RAW_CSV.exists():
        df = pd.read_csv(RAW_CSV)
        if "output_json" not in df.columns:
            raise ValueError(f"{RAW_CSV} has no `output_json` column and {RAW_JSONL} "
                             "is missing - nothing to parse.")
        recs = []
        for _, row in df.iterrows():
            try:
                result = json.loads(row["output_json"])
            except Exception:
                result = None
            recs.append({"filename": row["filename"],
                         "input_condition": row["input_condition"],
                         "model": row["model"], "result": result})
        print(f"Loaded {len(recs)} record(s) from {RAW_CSV} (output_json column)")
        return recs
    raise FileNotFoundError(f"Neither {RAW_JSONL} nor {RAW_CSV} found.")

records = load_records()


def gen_id(r):
    return f"{r['filename']}__{r['input_condition']}__{r['model']}"


def as_list(x):
    # Normalize a field that should be a list (None/scalar -> safe list).
    return x if isinstance(x, list) else ([] if x is None else [x])


def as_json(x):
    # Compact JSON string for multi-valued categories (None stays None/empty).
    return None if x is None else json.dumps(x, ensure_ascii=False)


def pick(items, field=None):
    # From a list of items (dicts or scalars), pull `field` from each dict
    # item (or the item itself when it is a scalar / field is None).
    out = []
    for it in as_list(items):
        out.append(it.get(field) if (isinstance(it, dict) and field is not None) else it)
    return out


def one_or_json(values):
    # A list of scalars -> a single scalar if there is one (non-null) value,
    # else a JSON list. Empty -> None.
    vals = [v for v in values if v is not None]
    if not vals:
        return None
    return vals[0] if len(vals) == 1 else json.dumps(vals, ensure_ascii=False)

Loaded 6 record(s) from moral_generation_plus.jsonl


## 3. Build the tidy tables

`morals_long` and `categories_wide` are built and written below. The five
per-dimension tables are written as commented-out blocks - un-comment a block
(and its write line) to also emit it.

In [5]:
morals = []
wide   = []

# --- OPTIONAL per-dimension accumulators (un-comment to use) ----------------
# dims, values, themes, actions, locations = [], [], [], [], []

for r in records:
    res = r.get("result")
    base = {"generation_id": gen_id(r), "filename": r["filename"],
            "input_condition": r["input_condition"], "model": r["model"]}
    if not isinstance(res, dict):
        continue  # unparsed generation -> nothing to flatten

    # === morals_long : one row per story moral (DEFAULT) ===
    for i, m in enumerate(as_list(res.get("story_morals")), 1):
        morals.append({**base, "moral_index": i, "moral_text": m})

    # === categories_wide : one row per generation, one column per category ===
    # Each category is reduced to a SINGLE chosen field. A field that yields
    # one value is written as a scalar; if it yields more than one, it is
    # written as a JSON list. story_morals is left as the full JSON array.
    prot = res.get("protagonist") or {}
    conf = res.get("central_conflict") or {}
    cons = res.get("consequence_resolution") or {}
    wide.append({**base,
        "protagonist":            prot.get("name"),
        "major_themes":           one_or_json(pick(res.get("major_themes"))),
        "major_locations":        one_or_json(pick(res.get("major_locations"), "location")),
        "central_conflict":       conf.get("thematic_opposing_force"),
        "principal_actions":      one_or_json(pick(res.get("principal_actions"), "action")),
        "values_pursued":         one_or_json(pick(res.get("values_pursued"), "value")),
        "values_threatened":      one_or_json(pick(res.get("values_threatened"), "value")),
        "consequence_resolution": cons.get("moral_significance"),
        "story_morals":           as_json(res.get("story_morals")),
    })

    # === OPTIONAL per-dimension tables (commented out) =======================
    # Un-comment the accumulator line above, a block here, and its write line
    # in the next cell.
    #
    # dimensions_wide -- the 1:1 scalar fields, one row per generation:
    # conf = res.get("central_conflict") or {}
    # cons = res.get("consequence_resolution") or {}
    # dims.append({**base,
    #     "protagonist_name": prot.get("name"),
    #     "protagonist_description": prot.get("description"),
    #     "protagonist_primary_goal": prot.get("primary_goal"),
    #     "central_conflict_summary": conf.get("summary"),
    #     "central_conflict_structural_type": conf.get("structural_type"),
    #     "central_conflict_thematic_opposing_force": conf.get("thematic_opposing_force"),
    #     "ending_summary": cons.get("ending_summary"),
    #     "moral_significance": cons.get("moral_significance")})
    #
    # values_long -- values pursued + threatened, one row per value:
    # for role, key in (("pursued","values_pursued"),("threatened","values_threatened")):
    #     for v in as_list(res.get(key)):
    #         vd = v if isinstance(v, dict) else {"value": v}
    #         values.append({**base, "value_role": role,
    #                        "value": vd.get("value"), "explanation": vd.get("explanation")})
    #
    # themes_long -- one row per theme:
    # for t in as_list(res.get("major_themes")):
    #     themes.append({**base, "theme": t})
    #
    # actions_long -- one row per principal action:
    # for i, a in enumerate(as_list(res.get("principal_actions")), 1):
    #     ad = a if isinstance(a, dict) else {"action": a}
    #     actions.append({**base, "action_index": i, "action": ad.get("action"),
    #                     "relationship_to_values_or_goals": ad.get("relationship_to_values_or_goals")})
    #
    # locations_long -- one row per location:
    # for l in as_list(res.get("major_locations")):
    #     ld = l if isinstance(l, dict) else {"location": l}
    #     locations.append({**base, "location": ld.get("location"),
    #                       "location_type": ld.get("location_type"),
    #                       "social_or_symbolic_significance": ld.get("social_or_symbolic_significance")})

# --- write the ACTIVE tables -------------------------------------------------
pd.DataFrame(morals, columns=["generation_id", "filename", "input_condition",
                              "model", "moral_index", "moral_text"]
             ).to_csv(OUTDIR / "morals_long.csv", index=False)

wide_cols = ["generation_id", "filename", "input_condition", "model", "protagonist",
             "major_themes", "major_locations", "central_conflict", "principal_actions",
             "values_pursued", "values_threatened", "consequence_resolution", "story_morals"]
pd.DataFrame(wide, columns=wide_cols).to_csv(OUTDIR / "categories_wide.csv", index=False)

print(f"morals_long     {len(morals):4d} rows -> {OUTDIR / 'morals_long.csv'}")
print(f"categories_wide {len(wide):4d} rows -> {OUTDIR / 'categories_wide.csv'}")

# --- OPTIONAL writes (commented out; pair with the blocks above) ------------
# pd.DataFrame(dims).to_csv(OUTDIR / "dimensions_wide.csv", index=False)
# pd.DataFrame(values).to_csv(OUTDIR / "values_long.csv", index=False)
# pd.DataFrame(themes).to_csv(OUTDIR / "themes_long.csv", index=False)
# pd.DataFrame(actions).to_csv(OUTDIR / "actions_long.csv", index=False)
# pd.DataFrame(locations).to_csv(OUTDIR / "locations_long.csv", index=False)

morals_long       18 rows -> tidy/morals_long.csv
categories_wide    6 rows -> tidy/categories_wide.csv


## 4. Peek at the outputs

In [ ]:
print("=== morals_long ===")
display(pd.read_csv(OUTDIR / "morals_long.csv").head(8))
print("=== categories_wide ===")
pd.read_csv(OUTDIR / "categories_wide.csv").head(4)